In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
import os
print(os.getcwd())

C:\Users\Pavillion\Documents\student-dropout-predictor\notebooks


In [6]:
import os
files = os.listdir('../data/')
print("Files in data folder:")
for f in files:
    print(f)

Files in data folder:
assessments.csv
courses.csv
result_by_age.png
result_by_disability.png
result_by_education.png
result_by_gender.png
studentAssessment.csv
studentInfo.csv
studentRegistration.csv
target_distribution.png
vle.csv


In [7]:
import os
files = os.listdir('../data/')
print("Files in data folder:")
for f in files:
    print(f)

Files in data folder:
assessments.csv
courses.csv
result_by_age.png
result_by_disability.png
result_by_education.png
result_by_gender.png
studentAssessment.csv
studentInfo.csv
studentRegistration.csv
studentVle.csv
target_distribution.png
vle.csv


In [8]:
# Loading all datasets
student_info = pd.read_csv('../data/studentInfo.csv')
student_assessment = pd.read_csv('../data/studentAssessment.csv')
student_vle = pd.read_csv('../data/studentVle.csv')
student_registration = pd.read_csv('../data/studentRegistration.csv')
assessments = pd.read_csv('../data/assessments.csv')

print("All datasets loaded!")
print(f"studentInfo shape: {student_info.shape}")

All datasets loaded!
studentInfo shape: (32593, 12)


In [ ]:
# We predict: Withdrawn = 1 (dropout), everything else = 0
student_info['dropout'] = (
    student_info['final_result'] == 'Withdrawn').astype(int)

print("Target variable created!")
print(student_info['dropout'].value_counts())
print(f"\nDropout rate: {student_info['dropout'].mean()*100:.1f}%")

## # Feature 1: Assessment scores per student

In [12]:
print("student_assessment columns:")
print(student_assessment.columns.tolist())
print()
print("assessments columns:")
print(assessments.columns.tolist())

student_assessment columns:
['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

assessments columns:
['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']


In [13]:
# First merge student_assessment with assessments to get code_module and code_presentation
assessment_merged = student_assessment.merge(
    assessments[['id_assessment', 'assessment_type', 
                 'weight', 'code_module', 'code_presentation']],
    on='id_assessment', how='left'
)

print("Merged columns:", assessment_merged.columns.tolist())

# Aggregate scores per student per module
assessment_features = assessment_merged.groupby(
    ['id_student', 'code_module', 'code_presentation']
).agg(
    avg_score=('score', 'mean'),
    max_score=('score', 'max'),
    min_score=('score', 'min'),
    total_assessments=('score', 'count'),
    submissions_on_time=('is_banked', 'sum')
).reset_index()

print("Assessment features created!")
print(assessment_features.shape)
print(assessment_features.head())

Merged columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score', 'assessment_type', 'weight', 'code_module', 'code_presentation']
Assessment features created!
(25843, 8)
   id_student code_module code_presentation  avg_score  max_score  min_score  \
0        6516         AAA             2014J  61.800000       77.0       48.0   
1        8462         DDD             2013J  87.666667       93.0       83.0   
2        8462         DDD             2014J  86.500000       93.0       83.0   
3       11391         AAA             2013J  82.000000       85.0       78.0   
4       23629         BBB             2013B  82.500000      100.0       63.0   

   total_assessments  submissions_on_time  
0                  5                    0  
1                  3                    0  
2                  4                    4  
3                  5                    0  
4                  4                    0  


## Feature 2: VLE (online activity) per student

In [14]:
# Aggregate VLE clicks per student
vle_features = student_vle.groupby(
    ['id_student', 'code_module', 'code_presentation']
).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    avg_clicks_per_day=('sum_click', 'mean')
).reset_index()

print("VLE features created!")
print(vle_features.shape)
print(vle_features.head())

VLE features created!
(29228, 6)
   id_student code_module code_presentation  total_clicks  active_days  \
0        6516         AAA             2014J          2791          159   
1        8462         DDD             2013J           646           56   
2        8462         DDD             2014J            10            1   
3       11391         AAA             2013J           934           40   
4       23629         BBB             2013B           161           16   

   avg_clicks_per_day  
0            4.216012  
1            2.153333  
2            2.500000  
3            4.765306  
4            2.728814  


## Feature 3: Registration features

In [15]:
# Days before course start that student registered
registration_features = student_registration.copy()
registration_features['registered_early'] = (
    registration_features['date_registration'] < 0).astype(int)

print("Registration features created!")
print(registration_features.head())

Registration features created!
  code_module code_presentation  id_student  date_registration  \
0         AAA             2013J       11391             -159.0   
1         AAA             2013J       28400              -53.0   
2         AAA             2013J       30268              -92.0   
3         AAA             2013J       31604              -52.0   
4         AAA             2013J       32885             -176.0   

   date_unregistration  registered_early  
0                  NaN                 1  
1                  NaN                 1  
2                 12.0                 1  
3                  NaN                 1  
4                  NaN                 1  


## Merge everything into one master dataset

In [16]:
# Start with student_info as base
master_df = student_info.copy()

# Merge assessment features
master_df = master_df.merge(
    assessment_features,
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

# Merge VLE features
master_df = master_df.merge(
    vle_features,
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

# Merge registration features
master_df = master_df.merge(
    registration_features[['id_student', 'code_module',
                           'code_presentation', 'date_registration',
                           'registered_early']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

print("Master dataset created!")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")

Master dataset created!
Shape: (32593, 23)
Columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'dropout', 'avg_score', 'max_score', 'min_score', 'total_assessments', 'submissions_on_time', 'total_clicks', 'active_days', 'avg_clicks_per_day', 'date_registration', 'registered_early']


In [17]:
print("Missing values before cleaning:")
print(master_df.isnull().sum()[master_df.isnull().sum() > 0])

# Filling missing numerical values with 0
# (students with no VLE activity or assessments)
fill_zero_cols = ['avg_score', 'max_score', 'min_score',
                  'total_assessments', 'submissions_on_time',
                  'total_clicks', 'active_days', 'avg_clicks_per_day',
                  'date_registration', 'registered_early']

for col in fill_zero_cols:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna(0)

print("\nMissing values after cleaning:")
print(master_df.isnull().sum()[master_df.isnull().sum() > 0])
print("\nAll good!" if master_df.isnull().sum().sum() == 0 
      else "Still some missing values to handle")

Missing values before cleaning:
imd_band               1111
avg_score              6773
max_score              6773
min_score              6773
total_assessments      6750
submissions_on_time    6750
total_clicks           3365
active_days            3365
avg_clicks_per_day     3365
date_registration        45
dtype: int64

Missing values after cleaning:
imd_band    1111
dtype: int64
Still some missing values to handle


In [18]:
# Encode categorical columns

from sklearn.preprocessing import LabelEncoder

# Columns to encode
cat_cols = ['code_module', 'code_presentation', 'gender',
            'region', 'highest_education', 'imd_band',
            'age_band', 'disability']

le = LabelEncoder()
for col in cat_cols:
    if col in master_df.columns:
        master_df[col + '_encoded'] = le.fit_transform(
            master_df[col].astype(str))

print("Categorical columns encoded!")
print("New encoded columns added:")
print([c for c in master_df.columns if '_encoded' in c])

Categorical columns encoded!
New encoded columns added:
['code_module_encoded', 'code_presentation_encoded', 'gender_encoded', 'region_encoded', 'highest_education_encoded', 'imd_band_encoded', 'age_band_encoded', 'disability_encoded']


In [19]:
# Select features for ML model
feature_cols = [
    # Encoded categoricals
    'gender_encoded', 'region_encoded', 'highest_education_encoded',
    'imd_band_encoded', 'age_band_encoded', 'disability_encoded',
    'code_module_encoded', 'code_presentation_encoded',
    # Student background
    'num_of_prev_attempts', 'studied_credits',
    # Assessment features
    'avg_score', 'max_score', 'min_score',
    'total_assessments', 'submissions_on_time',
    # VLE/online activity features
    'total_clicks', 'active_days', 'avg_clicks_per_day',
    # Registration
    'date_registration', 'registered_early'
]

# Target
target_col = 'dropout'

# Final dataset
X = master_df[feature_cols]
y = master_df[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {feature_cols}")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"\nDropout rate: {y.mean()*100:.1f}%")

Features shape: (32593, 20)
Target shape: (32593,)

Feature columns: ['gender_encoded', 'region_encoded', 'highest_education_encoded', 'imd_band_encoded', 'age_band_encoded', 'disability_encoded', 'code_module_encoded', 'code_presentation_encoded', 'num_of_prev_attempts', 'studied_credits', 'avg_score', 'max_score', 'min_score', 'total_assessments', 'submissions_on_time', 'total_clicks', 'active_days', 'avg_clicks_per_day', 'date_registration', 'registered_early']

Class distribution:
dropout
0    22437
1    10156
Name: count, dtype: int64

Dropout rate: 31.2%


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Split first — SMOTE only on training data!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nBefore SMOTE - Train class distribution:")
print(y_train.value_counts())

# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE - Train class distribution:")
print(pd.Series(y_train_smote).value_counts())

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

# Save everything for Phase 3
import pickle
with open('../data/X_train.pkl', 'wb') as f:
    pickle.dump(X_train_scaled, f)
with open('../data/X_test.pkl', 'wb') as f:
    pickle.dump(X_test_scaled, f)
with open('../data/y_train.pkl', 'wb') as f:
    pickle.dump(y_train_smote, f)
with open('../data/y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../data/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("\nAll data saved successfully!")

Train size: (26074, 20)
Test size:  (6519, 20)

Before SMOTE - Train class distribution:
dropout
0    17949
1     8125
Name: count, dtype: int64

After SMOTE - Train class distribution:
dropout
0    17949
1    17949
Name: count, dtype: int64

All data saved successfully!
